# 03 · Network Construction
**MLS NLP Pipeline — Stage 3**

Transforms enriched articles into weighted co-occurrence graphs.

### Two Graph Types

| Graph | Nodes | Edges |
|-------|-------|-------|
| **Club Co-occurrence** | MLS clubs | Both clubs mentioned in same article |
| **Club–Entity (bipartite)** | Clubs + persons | Club co-mentioned with player/coach/executive |

### Time Windows
- **Yearly** — 7 graphs (2018–2024)
- **Quarterly** — 28 graphs (e.g. `2022_Q4`)
- **Monthly** — 84 graphs (e.g. `2023_07`)

Edge weight = number of articles in which both nodes co-appear.

## 1. Setup

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from pipeline.utils import get_config, load_parquet, get_parquet_path, PROJECT_ROOT
from pipeline.network_builder import ClubCooccurrenceBuilder, ClubEntityBuilder, load_graph

settings  = get_config("settings")
data_dir  = PROJECT_ROOT / settings["pipeline"]["data_dir"]
net_cfg   = settings.get("networks", {})

print("Time windows       :", net_cfg.get("time_windows"))
print("Min edge weight    :", net_cfg.get("min_edge_weight"))
print("Excluded domains   :", net_cfg.get("excluded_domains"))
print("Max clubs/article  :", net_cfg.get("max_clubs_per_article"))

## 2. Building a Co-occurrence Graph — Step by Step

In [ ]:
# Load 2023 enriched data
frames = []
for month in range(1, 13):
    p = get_parquet_path(data_dir / "press" / "enriched", "enriched", 2023, month)
    df_m = load_parquet(p)
    if not df_m.empty:
        frames.append(df_m)

df_2023 = pd.concat(frames, ignore_index=True)
print(f"2023 enriched articles: {len(df_2023):,}")
df_2023[["clubs_mentioned", "primary_event_type", "sentiment_compound"]].head(5)

In [ ]:
# Apply domain filter (same as NetworkBuilderPipeline._filter_df)
excluded = set(net_cfg.get("excluded_domains", []))
max_clubs = net_cfg.get("max_clubs_per_article", 0)

before = len(df_2023)
if excluded and "domain" in df_2023.columns:
    df_2023 = df_2023[~df_2023["domain"].isin(excluded)]
if max_clubs > 0 and "club_mention_count" in df_2023.columns:
    df_2023 = df_2023[df_2023["club_mention_count"] <= max_clubs]

print(f"After filter: {len(df_2023):,} articles (removed {before - len(df_2023)})")

In [ ]:
# Build the co-occurrence graph
builder = ClubCooccurrenceBuilder(min_edge_weight=net_cfg.get("min_edge_weight", 2))
G = builder.build(df_2023)

print(f"Nodes : {G.number_of_nodes()}")
print(f"Edges : {G.number_of_edges()}")
print(f"Density: {nx.density(G):.4f}")

# Top 10 edges by weight
edges = sorted(G.edges(data=True), key=lambda e: e[2]["weight"], reverse=True)
print("\nTop 10 co-occurring club pairs (2023):")
for u, v, d in edges[:10]:
    print(f"  {u:<28} ↔ {v:<28}  weight={d['weight']}")

## 3. Visualising the 2023 Network

In [ ]:
CLUB_COLORS = {
    "Atlanta United FC": "#80000A", "Austin FC": "#00B140",
    "CF Montreal": "#003DA5",       "Charlotte FC": "#1A85C8",
    "Chicago Fire FC": "#CC0000",   "Colorado Rapids": "#862633",
    "Columbus Crew": "#FFF200",     "D.C. United": "#000000",
    "FC Dallas": "#BF0D3E",         "Houston Dynamo FC": "#F68712",
    "Inter Miami CF": "#F7B5CD",    "LA Galaxy": "#00245D",
    "LAFC": "#C39E6D",              "Minnesota United FC": "#8CD2F4",
    "Nashville SC": "#ECE83A",      "New England Revolution": "#0A2240",
    "New York City FC": "#6CACE4",  "New York Red Bulls": "#ED1E36",
    "Orlando City SC": "#612B9B",   "Philadelphia Union": "#071B2C",
    "Portland Timbers": "#00482B",  "Real Salt Lake": "#B30838",
    "San Jose Earthquakes": "#0D4C92", "Seattle Sounders FC": "#5D9741",
    "Sporting Kansas City": "#93B8E3", "St. Louis City SC": "#DD0000",
    "Toronto FC": "#B81137",        "Vancouver Whitecaps FC": "#009BC8",
}

pos = nx.spring_layout(G, seed=42, k=1.8)
node_colors = [CLUB_COLORS.get(n, "#888") for n in G.nodes()]
weights     = [G[u][v]["weight"] for u, v in G.edges()]
max_w       = max(weights)

fig, ax = plt.subplots(figsize=(14, 11))
ax.set_facecolor("#111111")

nx.draw_networkx_edges(G, pos, ax=ax,
    width=[0.5 + 3.5 * w / max_w for w in weights],
    alpha=[0.2 + 0.7 * w / max_w for w in weights],
    edge_color="white")

nx.draw_networkx_nodes(G, pos, ax=ax,
    node_color=node_colors,
    node_size=[300 + 800 * G.degree(n, weight="weight") / max(dict(G.degree(weight="weight")).values())
               for n in G.nodes()])

labels = {n: n.replace(" FC", "").replace(" CF", "").replace(" SC", "") for n in G.nodes()}
nx.draw_networkx_labels(G, pos, labels, ax=ax, font_size=7, font_color="white")

ax.set_title("MLS Club Co-occurrence Network — 2023
(edge weight = articles co-mentioning both clubs)",
             color="white", fontsize=13, fontweight="bold")
ax.set_axis_off()
fig.patch.set_facecolor("#111111")
plt.tight_layout()
plt.show()

## 4. Bipartite Club–Entity Graph

In [ ]:
entity_builder = ClubEntityBuilder(min_edge_weight=2)
G_bi = entity_builder.build(df_2023)

clubs   = [n for n, d in G_bi.nodes(data=True) if d.get("entity_type") == "club"]
players = [n for n, d in G_bi.nodes(data=True) if d.get("entity_type") == "player"]
coaches = [n for n, d in G_bi.nodes(data=True) if d.get("entity_type") == "coach"]

print(f"Club nodes   : {len(clubs)}")
print(f"Player nodes : {len(players)}")
print(f"Coach nodes  : {len(coaches)}")
print(f"Edges        : {G_bi.number_of_edges()}")

# Top connected persons
person_degrees = {n: G_bi.degree(n, weight="weight")
                  for n in G_bi.nodes()
                  if G_bi.nodes[n].get("entity_type") != "club"}
top_persons = sorted(person_degrees.items(), key=lambda x: x[1], reverse=True)[:15]
print("\nTop 15 most-mentioned persons (2023):")
for person, deg in top_persons:
    role = G_bi.nodes[person].get("entity_type", "?")
    print(f"  {person:<28} {role:<12} weight={deg}")

## 5. Network Statistics Across Years

In [ ]:
net_dir = data_dir / "press" / "networks"
stats = []
for year in range(2018, 2025):
    json_path = net_dir / str(year) / f"{year}_club_cooccurrence.json"
    if json_path.exists():
        G_y = load_graph(json_path)
        stats.append({
            "year": year,
            "nodes": G_y.number_of_nodes(),
            "edges": G_y.number_of_edges(),
            "density": round(nx.density(G_y), 4),
            "avg_weight": round(sum(d["weight"] for _, _, d in G_y.edges(data=True)) / max(G_y.number_of_edges(), 1), 1),
        })

pd.DataFrame(stats).set_index("year")

## 6. How Saved Graph Files Are Structured

In [ ]:
# Peek at the JSON format (node-link)
sample_json = net_dir / "2023" / "2023_club_cooccurrence.json"
with open(sample_json) as f:
    data = json.load(f)

print("Keys in JSON:", list(data.keys()))
print("\nSample node:", data["nodes"][0])
print("Sample link:", data["links"][0])